In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

d:\Data Science\ML Ops Youtube Anas Riad - 1-Sep-26\Regression_ML_EndtoEnd\.venv\Scripts\python.exe
3.0.4
d:\Data Science\ML Ops Youtube Anas Riad - 1-Sep-26\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

d:\Data Science\ML Ops Youtube Anas Riad - 1-Sep-26\Regression_ML_EndtoEnd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("D:/Data Science/ML Ops Youtube Anas Riad - 1-Sep-26/Regression_ML_EndtoEnd/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("D:/Data Science/ML Ops Youtube Anas Riad - 1-Sep-26/Regression_ML_EndtoEnd/data/processed/feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (585199, 39)
Eval shape: (148448, 39)


In [6]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [8]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("file:///D:/Data Science/ML Ops Youtube Anas Riad - 1-Sep-26/Regression_ML_EndtoEnd/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/09/05 17:35:07 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-09-05 17:35:07,359] A new study created in memory with name: no-name-876112c7-72d1-400d-9a15-33e0a8d462c0
[I 2026-09-05 17:35:36,757] Trial 0 finished with value: 78196.93081919894 and parameters: {'n_estimators': 921, 'max_depth': 4, 'learning_rate': 0.020340601854533423, 'subsample': 0.533288315164151, 'colsample_bytree': 0.9572880011780918, 'min_child_weight': 2, 'gamma': 2.0529633451799825, 'reg_alpha': 9.830361650875972, 'reg_lambda': 0.0012080428545628305}. Best is trial 0 with value: 78196.93081919894.
[I 2026-09-05 17:35:59,769] Trial 1 finished with value: 77113.72668730558 and parameters: {'n_estimators': 251, 'max_depth': 8, 'learning_rate': 0.019864537613107146, 'subsample': 0.858474894301936, 'colsample_bytree': 0.8807642315402237, 'min_child_weight': 5, 'gamma': 1.4027314847078158, 'reg_alpha': 2.4589057770139562e-08, 'reg_lambd

Best params: {'n_estimators': 768, 'max_depth': 7, 'learning_rate': 0.05766897585181406, 'subsample': 0.987473685045175, 'colsample_bytree': 0.5082532629081182, 'min_child_weight': 7, 'gamma': 4.95241088753016, 'reg_alpha': 6.066073367411233e-07, 'reg_lambda': 1.2827802024451917e-05}


In [ ]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 31400.608307640367
RMSE: 71322.88512765609
R²: 0.9606886610012438


d:\Data Science\ML Ops Youtube Anas Riad - 1-Sep-26\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [17:46:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/09/05 17:46:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


: 